# Exercise 4.4 – Local Search Algorithms
## ICOM 5016 – Artificial Intelligence

---

## 1. Introduction

This report presents the implementation and experimental evaluation of local search algorithms applied to two classic AI problems: the **8-Queens problem** and the **8-Puzzle problem**.

Local search algorithms operate on a single current state and move to neighboring states. Unlike systematic search, they do not keep track of paths and are particularly useful when the goal state itself is the solution (not the path to it). The algorithms evaluated are:

- **Steepest-Ascent Hill Climbing** – At each step, evaluates all neighbors and selects the one with the highest heuristic value. Gets stuck easily at local maxima.
- **First-Choice Hill Climbing** – Generates random successors until one better than the current state is found. More efficient in high-branching-factor spaces.
- **Hill Climbing with Random Restart** – Repeats hill climbing from randomly generated initial states until a solution is found. Trivially complete given enough restarts.
- **Simulated Annealing** – Probabilistically accepts worse states early on (controlled by a temperature schedule), gradually becoming greedier. Can escape local maxima.

For each algorithm and problem, we measure the **success rate** (percentage of instances solved) and the **average search cost** (number of steps taken). Results are compared against the optimal solution cost obtained via A* search.

The implementation uses the UC Berkeley AIMA Python repository [1].

---
## 2. Setup and Imports

We import everything from the AIMA `search` module, which provides the base `Problem` class, `Node`, `hill_climbing`, `simulated_annealing`, `astar_search`, and `EightPuzzle`. We also import `random`, `numpy`, and `matplotlib` for experiments and visualization.

In [ ]:
from search import *
import random
import numpy as np
import matplotlib.pyplot as plt

---
## 3. Problem Definitions

### 3.1 The 8-Queens Problem

The 8-Queens problem asks us to place 8 queens on an 8×8 chessboard such that no two queens attack each other (no two queens share the same row, column, or diagonal).

**State representation:** A tuple of 8 integers where `state[i]` is the row of the queen in column `i`. This ensures no two queens are ever in the same column.

**Actions:** Move any queen to a different row in its column.

**Heuristic value (`value`):** We want to *maximize* the number of non-attacking pairs. The maximum possible non-attacking pairs for 8 queens is `8*(8-1)/2 = 28`. So `value = 28 - conflicts`.

**Goal:** Zero conflicts (value = 28).

In [ ]:
class QueensLocalSearch(Problem):
    """8-Queens problem formulated for local search.
    State: tuple of 8 ints, where state[col] = row of queen in that column.
    Value: number of non-attacking pairs (maximize).
    """

    def __init__(self, n=8):
        self.n = n
        # Start from a random initial placement
        initial = tuple(random.randint(0, n - 1) for _ in range(n))
        super().__init__(initial)

    def actions(self, state):
        """All possible moves: move any queen to any other row in its column."""
        return [
            (col, row)
            for col in range(self.n)
            for row in range(self.n)
            if row != state[col]
        ]

    def result(self, state, action):
        """Apply action: move the queen in `col` to `row`."""
        col, row = action
        new_state = list(state)
        new_state[col] = row
        return tuple(new_state)

    def conflicts(self, state):
        """Count the number of attacking pairs of queens."""
        conflicts = 0
        for i in range(self.n):
            for j in range(i + 1, self.n):
                same_row = state[i] == state[j]
                same_diagonal = abs(state[i] - state[j]) == abs(i - j)
                if same_row or same_diagonal:
                    conflicts += 1
        return conflicts

    def goal_test(self, state):
        """Goal is zero conflicts."""
        return self.conflicts(state) == 0

    def value(self, state):
        """Heuristic: maximize non-attacking pairs."""
        max_pairs = self.n * (self.n - 1) // 2
        return max_pairs - self.conflicts(state)

### 3.2 The 8-Puzzle Problem

The 8-Puzzle consists of a 3×3 grid with 8 numbered tiles and one blank space. The goal is to slide tiles into the configuration `(1,2,3,4,5,6,7,8,0)` where `0` is the blank.

**State representation:** A 9-tuple where `state[i]` is the tile at position `i` (row-major order). `0` represents the blank.

**Heuristic value (`value`):** We use the **negative Manhattan distance**: The sum of distances of each tile from its goal position. Since local search *maximizes* value, a value of 0 means the goal is reached.

**Random puzzle generation:** We generate random puzzles by starting from the goal state and performing 50 random legal moves. This guarantees solvability.

**Optimal cost:** Obtained via A* search with Manhattan distance heuristic, used as a baseline for comparison.

In [ ]:
class PuzzleWithValue(EightPuzzle):
    """Extends the AIMA EightPuzzle to add a value function for local search.
    Value = negative Manhattan distance (0 at goal, more negative when farther).
    """

    def value(self, state):
        """Negative total Manhattan distance of all tiles from their goal positions."""
        distance = 0
        goal = (1, 2, 3, 4, 5, 6, 7, 8, 0)
        for i in range(9):
            if state[i] == 0:
                continue  # Blank tile doesn't count
            goal_index = goal.index(state[i])
            x1, y1 = divmod(i, 3)
            x2, y2 = divmod(goal_index, 3)
            distance += abs(x1 - x2) + abs(y1 - y2)
        return -distance


def random_puzzle(moves=50):
    """Generate a random solvable puzzle by shuffling from the goal state."""
    problem = EightPuzzle((1, 2, 3, 4, 5, 6, 7, 8, 0))
    state = problem.initial
    for _ in range(moves):
        actions = problem.actions(state)
        state = problem.result(state, random.choice(actions))
    return state


def get_optimal_cost(state):
    """Use A* search to find the optimal number of moves to solve the puzzle."""
    problem = EightPuzzle(state)
    result = astar_search(problem)
    if result is None:
        return None
    return len(result.solution())

---
## 4. Algorithm Implementations

### 4.1 Steepest-Ascent Hill Climbing (with cost tracking)

The AIMA `hill_climbing` function implements steepest-ascent but does not track search cost. We wrap it here to also count the number of state evaluations (steps).

At each iteration, we expand all neighbors (cost += number of neighbors evaluated), then move to the best one, or stop if no neighbor improves the current value.

In [ ]:
def steepest_hc_with_cost(problem):
    """Steepest-ascent hill climbing.
    Returns: (final_state, steps)
    steps = total number of neighbor evaluations.
    """
    current = Node(problem.initial)
    steps = 0
    while True:
        neighbors = current.expand(problem)
        steps += len(neighbors)  # We evaluated all neighbors this round
        if not neighbors:
            return current.state, steps
        # Pick the neighbor with the highest value
        best = argmax_random_tie(neighbors, key=lambda node: problem.value(node.state))
        if problem.value(best.state) <= problem.value(current.state):
            # No improvement — local maximum reached
            return current.state, steps
        current = best

### 4.2 First-Choice Hill Climbing

Instead of evaluating *all* neighbors, first-choice hill climbing randomly shuffles the actions and tries them one by one until it finds one that improves the current state. This is more efficient when the branching factor is large.

For the 8-Queens problem, each state has 8×7 = 56 possible moves: steepest-ascent evaluates all 56 every step, while first-choice stops as soon as it finds an improvement.

Search cost = number of successor states evaluated before finding an improvement (or stopping).

In [ ]:
def first_choice_hc(problem):
    """First-choice hill climbing.
    Randomly tries successors until one improves the current state.
    Returns: (final_state, steps)
    """
    current = Node(problem.initial)
    steps = 0
    while True:
        actions = list(problem.actions(current.state))
        random.shuffle(actions)  # Try actions in random order
        improved = False
        for action in actions:
            neighbor_state = problem.result(current.state, action)
            steps += 1
            if problem.value(neighbor_state) > problem.value(current.state):
                current = Node(neighbor_state)
                improved = True
                break  # Stop as soon as we find an improvement
        if not improved:
            # No improving neighbor found — local maximum
            return current.state, steps

### 4.3 Random Restart Hill Climbing

Random restart runs steepest-ascent hill climbing from a new random initial state each time it gets stuck. It keeps restarting until a solution is found or a maximum number of restarts is reached.

Search cost = total steps across all restarts combined.

In [ ]:
def random_restart_hc(problem_class, max_restarts=20, **kwargs):
    """Hill climbing with random restart.
    Creates a fresh problem instance each restart (new random initial state).
    Returns: (solved, total_steps)
    """
    total_steps = 0
    for _ in range(max_restarts):
        problem = problem_class(**kwargs)
        result, cost = steepest_hc_with_cost(problem)
        total_steps += cost
        if problem.goal_test(result):
            return True, total_steps
    return False, total_steps

### 4.4 Simulated Annealing Schedules

Simulated annealing uses a temperature schedule that controls how likely the algorithm is to accept a worse state. At high temperatures, worse moves are accepted often (exploration). As temperature decreases, the algorithm becomes greedier (exploitation).

We define exponential cooling schedules for each problem:
- `T(t) = T0 * alpha^t`: temperature decreases exponentially at each step
- When `T` drops below a threshold (or time limit `t_max`), we return 0 (no more exploration)

In [ ]:
def queens_schedule(t):
    """Annealing schedule for 8-Queens: starts hot, cools over 300 steps."""
    if t > 300:
        return 0
    return 20 * (0.95 ** t)


def puzzle_schedule(t):
    """Annealing schedule for 8-Puzzle: cooler start, cools over 300 steps."""
    if t > 300:
        return 0
    return 10 * (0.95 ** t)

---
## 5. Experiments

### 5.1 8-Queens Experiments

We run 200 independent trials for each algorithm. In each trial:
- A new random 8-Queens instance is generated
- The algorithm is run on it
- We record whether it solved the problem and how many steps it took

For **Random Restart**, we allow up to 20 restarts per trial and sum the total steps across all restarts.

In [ ]:
NUM_QUEENS_TRIALS = 200

queens_data = {
    "steepest":       {"solved": 0, "costs": []},
    "first_choice":   {"solved": 0, "costs": []},
    "random_restart": {"solved": 0, "costs": []},
    "annealing":      {"solved": 0, "costs": []},
}

for _ in range(NUM_QUEENS_TRIALS):

    # --- Steepest-Ascent ---
    p = QueensLocalSearch(8)
    result, cost = steepest_hc_with_cost(p)
    queens_data["steepest"]["costs"].append(cost)
    if p.goal_test(result):
        queens_data["steepest"]["solved"] += 1

    # --- First-Choice ---
    p = QueensLocalSearch(8)
    result, cost = first_choice_hc(p)
    queens_data["first_choice"]["costs"].append(cost)
    if p.goal_test(result):
        queens_data["first_choice"]["solved"] += 1

    # --- Random Restart ---
    solved, cost = random_restart_hc(QueensLocalSearch, max_restarts=20, n=8)
    queens_data["random_restart"]["costs"].append(cost)
    if solved:
        queens_data["random_restart"]["solved"] += 1

    # --- Simulated Annealing ---
    p = QueensLocalSearch(8)
    result = simulated_annealing(p, schedule=queens_schedule)
    queens_data["annealing"]["costs"].append(300)  # SA runs fixed schedule length
    if p.goal_test(result):
        queens_data["annealing"]["solved"] += 1

print("8-Queens experiments complete.")

### 5.2 8-Puzzle Experiments

We run 100 independent trials. In each trial:
- A new random solvable 8-Puzzle instance is generated
- A* search finds the **optimal solution cost** (used as a baseline)
- All four local search algorithms are run on the same instance
- We record whether each algorithm solved it and the search cost

Note: For **Random Restart** on the 8-Puzzle, we restart from the *same* initial state (since the puzzle is fixed), not from a random one but restarts don't help much here since the state space is determined by the initial configuration.

In [ ]:
NUM_PUZZLE_TRIALS = 100

puzzle_data = []

for _ in range(NUM_PUZZLE_TRIALS):
    state = random_puzzle(moves=50)
    optimal = get_optimal_cost(state)
    if optimal is None:
        continue

    # --- Steepest-Ascent ---
    p = PuzzleWithValue(state)
    hc_result, hc_cost = steepest_hc_with_cost(p)
    hc_solved = p.goal_test(hc_result)

    # --- First-Choice ---
    p = PuzzleWithValue(state)
    fc_result, fc_cost = first_choice_hc(p)
    fc_solved = p.goal_test(fc_result)

    # --- Simulated Annealing ---
    p = PuzzleWithValue(state)
    sa_result = simulated_annealing(p, schedule=puzzle_schedule)
    sa_solved = p.goal_test(sa_result)

    # --- Random Restart (up to 10 restarts, same start state) ---
    rr_solved = False
    rr_cost = 0
    for _ in range(10):
        p = PuzzleWithValue(state)
        rr_result, cost = steepest_hc_with_cost(p)
        rr_cost += cost
        if p.goal_test(rr_result):
            rr_solved = True
            break

    puzzle_data.append({
        "optimal": optimal,
        "hc_solved": hc_solved, "hc_cost": hc_cost,
        "fc_solved": fc_solved, "fc_cost": fc_cost,
        "sa_solved": sa_solved,
        "rr_solved": rr_solved, "rr_cost": rr_cost,
    })

print(f"8-Puzzle experiments complete. Valid trials: {len(puzzle_data)}")

---
## 6. Results and Analysis

### 6.1 8-Queens Results

Table 1 summarizes the success rate and average search cost for each algorithm on the 8-Queens problem over 200 trials. Figures 1 and 2 visualize these results.

In [ ]:
alg_keys   = ["steepest", "first_choice", "random_restart", "annealing"]
alg_labels = ["Steepest-Ascent HC", "First-Choice HC", "Random Restart HC", "Simulated Annealing"]

q_rates = [queens_data[k]["solved"] / NUM_QUEENS_TRIALS for k in alg_keys]
q_costs = [np.mean(queens_data[k]["costs"]) for k in alg_keys]

print("Table 1. 8-Queens Results (200 trials)")
print(f"{'Algorithm':<25} {'Success Rate':>14} {'Avg Search Cost':>17}")
print("-" * 58)
for label, rate, cost in zip(alg_labels, q_rates, q_costs):
    print(f"{label:<25} {rate:>13.1%} {cost:>17.1f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

axes[0].bar(alg_labels, q_rates, color=colors)
axes[0].set_title("Figure 1. 8-Queens: Success Rate per Algorithm")
axes[0].set_ylabel("Success Rate")
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis='x', rotation=20)
for i, v in enumerate(q_rates):
    axes[0].text(i, v + 0.02, f"{v:.0%}", ha='center', fontsize=10)

axes[1].bar(alg_labels, q_costs, color=colors)
axes[1].set_title("Figure 2. 8-Queens: Average Search Cost per Algorithm")
axes[1].set_ylabel("Average Steps")
axes[1].tick_params(axis='x', rotation=20)
for i, v in enumerate(q_costs):
    axes[1].text(i, v + 1, f"{v:.0f}", ha='center', fontsize=10)

plt.tight_layout()
plt.show()

**Analysis of 8-Queens Results:**

As shown in Table 1 and Figures 1 and 2, the algorithms differ significantly in both success rate and search cost:

- **Steepest-Ascent Hill Climbing** has the lowest success rate, typically around 14%. It gets stuck at local maxima frequently, states where all neighboring configurations have equal or worse values. Once stuck, it cannot escape.

- **First-Choice Hill Climbing** achieves a similar success rate to steepest-ascent, but with significantly lower average search cost because it stops evaluating neighbors as soon as an improvement is found (instead of checking all 56 neighbors).

- **Random Restart Hill Climbing** has the highest success rate, typically above 90%. By repeatedly restarting from different random positions, it effectively samples many different regions of the state space, making it likely that at least one restart lands near a solution basin.

- **Simulated Annealing** achieves moderate success. It can occasionally escape local maxima, but the solution quality is sensitive to the temperature schedule. A poorly tuned schedule may cause it to accept too many bad moves or cool too quickly.

### 6.2 8-Puzzle Results

Table 2 summarizes the 8-Puzzle results. Figures 3 and 4 show success rates and average search cost compared to the A* optimal. Note that A* always solves the puzzle optimally, so its success rate is 100% and its cost represents the true minimum.

In [ ]:
n = len(puzzle_data)

hc_rate  = sum(1 for r in puzzle_data if r["hc_solved"]) / n
fc_rate  = sum(1 for r in puzzle_data if r["fc_solved"]) / n
sa_rate  = sum(1 for r in puzzle_data if r["sa_solved"]) / n
rr_rate  = sum(1 for r in puzzle_data if r["rr_solved"]) / n

hc_avg_cost  = np.mean([r["hc_cost"] for r in puzzle_data])
fc_avg_cost  = np.mean([r["fc_cost"] for r in puzzle_data])
rr_avg_cost  = np.mean([r["rr_cost"] for r in puzzle_data])
opt_avg_cost = np.mean([r["optimal"] for r in puzzle_data])

print("Table 2. 8-Puzzle Results (100 trials)")
print(f"{'Algorithm':<25} {'Success Rate':>14} {'Avg Search Cost':>17}")
print("-" * 58)
rows = [
    ("Steepest-Ascent HC",   hc_rate,  hc_avg_cost),
    ("First-Choice HC",      fc_rate,  fc_avg_cost),
    ("Random Restart HC",    rr_rate,  rr_avg_cost),
    ("Simulated Annealing",  sa_rate,  None),
    ("A* (Optimal Baseline)",1.0,      opt_avg_cost),
]
for label, rate, cost in rows:
    cost_str = f"{cost:>17.1f}" if cost is not None else f"{'N/A':>17}"
    print(f"{label:<25} {rate:>13.1%} {cost_str}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

puz_labels = ["Steepest HC", "First-Choice HC", "Random Restart", "Sim. Annealing"]
puz_rates  = [hc_rate, fc_rate, rr_rate, sa_rate]
puz_costs  = [hc_avg_cost, fc_avg_cost, rr_avg_cost]
cost_labels = ["Steepest HC", "First-Choice HC", "Random Restart", "A* Optimal"]
cost_vals   = [hc_avg_cost, fc_avg_cost, rr_avg_cost, opt_avg_cost]
cost_colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

axes[0].bar(puz_labels, puz_rates, color=colors)
axes[0].set_title("Figure 3. 8-Puzzle: Success Rate per Algorithm")
axes[0].set_ylabel("Success Rate")
axes[0].set_ylim(0, 1.05)
axes[0].tick_params(axis='x', rotation=20)
for i, v in enumerate(puz_rates):
    axes[0].text(i, v + 0.02, f"{v:.0%}", ha='center', fontsize=10)

axes[1].bar(cost_labels, cost_vals, color=cost_colors)
axes[1].set_title("Figure 4. 8-Puzzle: Avg Search Cost vs A* Optimal")
axes[1].set_ylabel("Average Steps")
axes[1].tick_params(axis='x', rotation=20)
for i, v in enumerate(cost_vals):
    axes[1].text(i, v + 0.3, f"{v:.1f}", ha='center', fontsize=10)

plt.tight_layout()
plt.show()

**Analysis of 8-Puzzle Results:**

As shown in Table 2 and Figures 3 and 4, local search algorithms perform poorly on the 8-Puzzle compared to A* search:

- **Steepest-Ascent and First-Choice Hill Climbing** solve a small fraction of instances. The 8-Puzzle's landscape is full of local maxima and plateaus where the Manhattan distance cannot be improved by any single tile move. Once stuck, the algorithm cannot recover.

- **Random Restart** improves slightly over single-run hill climbing, but since the initial state is fixed (the puzzle is given), restarts do not help as much as in the 8-Queens case.

- **Simulated Annealing** is also limited. The 8-Puzzle requires a very precise sequence of moves, and random perturbations rarely lead to solutions without a carefully tuned schedule.

- **A* search**, as shown in Figure 4, solves every instance optimally with a much lower average move count. This demonstrates that local search is fundamentally unsuited to the 8-Puzzle — path-based search is necessary when the solution requires a specific sequence of moves.

---
## 7. Conclusions

This experiment demonstrates the strengths and limitations of local search algorithms:

1. **Problem structure matters.** Local search works well on problems like 8-Queens where the goal is a configuration rather than a path. It fails on problems like 8-Puzzle where a precise sequence is required.

2. **Random Restart is highly effective** when success from any single random start has a reasonable probability. With enough restarts, it becomes practically complete.

3. **First-Choice Hill Climbing is more efficient** than Steepest-Ascent in terms of search cost, as it avoids evaluating all neighbors when an improvement can be found quickly.

4. **Simulated Annealing** can escape local maxima but is sensitive to schedule tuning. A poorly tuned schedule can perform worse than simple hill climbing.

5. **For puzzles requiring optimal paths**, classical search algorithms like A* remain superior.

---
## 8. References

[1] S. Russell and P. Norvig, *Artificial Intelligence: A Modern Approach*, 4th ed. Hoboken, NJ: Pearson, 2020. [Online]. Available: https://github.com/aimacode/aima-python

[2] N. G. Santiago and M. A. Jiménez, *Writing Formal Reports: An Approach for Engineering Students in 21st Century*, 3rd ed. Mayagüez, PR, 2002. [Online]. Available: https://ece.uprm.edu/~hunt/inel5326/ReporteFinal.pdf. [Accessed: Apr. 27, 2025].

[3] IEEEDataPort, "How to Cite References: IEEE Documentation Style," [Online]. Available: https://ieee-dataport.org/sites/default/files/analysis/27/IEEE%20Citation%20Guidelines.pdf. [Accessed: Apr. 27, 2025].